In [8]:
library(Seurat)
# library(SeuratDisk)

library(reticulate)
library(anndata)

library(ggplot2)
library(ggpubr)
library(pheatmap)
library(dplyr)
library(tidyr)
library(RColorBrewer)
library(clustree)

library(UpSetR)
library(grid)

library(PRROC)
library(Matrix)

getwd()

dataset_id <- "simulated_mm_RA"
genome_id <- "mm10"
samples <- c("all", "old", "young")

for (sample in samples) {
    dir.create(paste0("figures_", dataset_id, "_", sample))
}

colorTools <- c(
    "STARsolo_TE" = "#FFBC81",
    "STARsolo_TE_EM" = "#f0df93",
    "SoloTE_unique" = "#A4DD9B",
    "SoloTE_thr2" = "#6FC69D",
    "SoloTE_thr1" = "#4BB2BB",
    "SoloTE_thr0" = "#3989BF",
    "Stellarscope" = "#4F5D93",
    "simulated" = "grey70"
)

thrMinCells <- 500 * 0.05

[1] "/mnt/TEdata/TEbenchmarking/Rscripts"

Warning message in dir.create(paste0("figures_", dataset_id, "_", sample)):
“'figures_simulated_mm_RA_all' already exists”
Warning message in dir.create(paste0("figures_", dataset_id, "_", sample)):
“'figures_simulated_mm_RA_old' already exists”
Warning message in dir.create(paste0("figures_", dataset_id, "_", sample)):
“'figures_simulated_mm_RA_young' already exists”


In [2]:
# import TE annotation with conversion columns for SoloTE and Stellarscope
conversion_table <- read.table(paste0("annotation/annotation_", genome_id, "_conversion_withAge.tsv"))
conversion_table$soloteID <- gsub("\\?", "", conversion_table$soloteID)

Create list of objects with counts generated by each colorTools

In [3]:
objList <- list()

objGeneList <- list()


# Splatter simulation

In [5]:
splatter_objs <- list()
samples <- c("all", "old", "young")

# path where simulated matrices are stored
main_path <- paste0("simulatedData/minnow_simulate_mm_RA_TE_")

for (sample in samples) {
    print(sample)

    path <- paste0(main_path, sample, "/")

    splatter_mat <- read.table(paste0(path, "quants_mat.csv"),
                               sep = ",", header = FALSE)

    # remove NAs, which are present because rows end with ","
    splatter_mat <- splatter_mat %>% select_if(~ !all(is.na(.)))
    colnames(splatter_mat) <- read.table(paste0(path, "quants_mat_cols.txt"))$V1
    rownames(splatter_mat) <- read.table(paste0(path, "quants_mat_rows.txt"))$V1
    # transpose matrix
    splatter_mat <- t(splatter_mat)

    print("loci in conversion table:")
    print(table(rownames(splatter_mat) %in% conversion_table$locusID))

    rownames(splatter_mat) <- gsub(pattern = "_", replacement = "-", rownames(splatter_mat))

    splatter_objs[[sample]] <- Seurat::CreateSeuratObject(splatter_mat,
        project = paste0("simulated_", sample),
        min.cells = thrMinCells, min.features = 0
    )
    print(splatter_objs[[sample]])
    # around 100 features are removed because expressed in less than the threshold of cells
    print("loci matching stellarscope ID:")
    print(table(rownames(splatter_objs[[sample]]) %in% conversion_table$stellarscopeID))

    print("Summary of n counts")
    print(summary(splatter_objs[[sample]]$nCount_RNA))
    print("Summary of n features")
    print(summary(splatter_objs[[sample]]$nFeature_RNA))
}

[1] "all"
[1] "loci in conversion table:"

TRUE 
5273 


Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


An object of class Seurat 
4947 features across 500 samples within 1 assay 
Active assay: RNA (4947 features, 0 variable features)
 1 layer present: counts
[1] "loci matching stellarscope ID:"

TRUE 
4947 
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  28658   52253   59826   60754   68977  112605 
[1] "Summary of n features"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2819    3408    3528    3521    3649    4036 
[1] "old"
[1] "loci in conversion table:"

TRUE 
3173 


Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


An object of class Seurat 
3030 features across 500 samples within 1 assay 
Active assay: RNA (3030 features, 0 variable features)
 1 layer present: counts
[1] "loci matching stellarscope ID:"

TRUE 
3030 
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  28191   52205   59970   60769   68261  113835 
[1] "Summary of n features"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2018    2294    2360    2353    2415    2613 
[1] "young"
[1] "loci in conversion table:"

TRUE 
2100 


Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


An object of class Seurat 
2030 features across 500 samples within 1 assay 
Active assay: RNA (2030 features, 0 variable features)
 1 layer present: counts
[1] "loci matching stellarscope ID:"

TRUE 
2030 
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  27752   52153   60062   60785   68740  112572 
[1] "Summary of n features"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   1428    1634    1668    1665    1702    1816 


## Normalize splatter

In [6]:
for (sample in samples) {
    splatter_objs[[sample]] <- NormalizeData(splatter_objs[[sample]],
        normalization.method = "LogNormalize",
        scale.factor = 10000
    )
}

Normalizing layer: counts



Normalizing layer: counts

Normalizing layer: counts



# STARsolo TE 

In [18]:
STARsolo_TE_objs <- list()
samples <- c("all", "old", "young")

STARsolo_TE_path <- ""
main_path <- paste0(STARsolo_TE_path, "/", dataset_id, "/")

for (sample in samples) {
    print(sample)

    path <- paste0(main_path, sample, "/TE_Solo.out/Gene/raw/")

    filteredBarcodes <- read.table(paste0(main_path, sample, "/TE_Solo.out/Gene/filtered/barcodes.tsv"))$V1

    STARsolo_TE_mat <- Seurat::ReadMtx(
        mtx = paste0(path, "matrix.mtx"),
        cells = paste0(path, "barcodes.tsv"),
        features = paste0(path, "features.tsv")
    ) # read matrix
    STARsolo_TE_mat <- STARsolo_TE_mat[, filteredBarcodes] # keep only barcodes passing thresholds

    STARsolo_TE_objs[[sample]] <- Seurat::CreateSeuratObject(STARsolo_TE_mat,
        project = paste0("STARsolo_TE_", sample),
        min.cells = thrMinCells, min.features = 0
    )

    # Remove some classes of TEs (already not present in SoloTE)
    classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")
    starsoloTEs <- Features(STARsolo_TE_objs[[sample]])

    filteredStarsoloTEs <- conversion_table[conversion_table$stellarscopeID %in% starsoloTEs, ] %>%
        dplyr::filter(!class %in% classesToExclude) %>%
        pull(stellarscopeID)

    print("Percentage of TEs in removed orders:")
    print(length(setdiff(starsoloTEs, filteredStarsoloTEs)) / length(starsoloTEs) * 100)

    STARsolo_TE_objs[[sample]] <- STARsolo_TE_objs[[sample]][intersect(starsoloTEs, filteredStarsoloTEs), ]

    print("Summary of n counts")
    summary(STARsolo_TE_objs[[sample]]$nCount_RNA)
    print("Summary of n features")
    summary(STARsolo_TE_objs[[sample]]$nFeature_RNA)

    print("Loci present in the annotation:")
    print(table(rownames(STARsolo_TE_objs[[sample]]) %in% conversion_table$stellarscopeID))
}

# add to the list of objects
objList[["STARsolo_TE"]] <- STARsolo_TE_objs


[1] "all"


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "Percentage of TEs in removed orders:"
[1] 0
[1] "Summary of n counts"
[1] "Summary of n features"
[1] "Loci present in the annotation:"

 TRUE 
14107 
[1] "old"


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "Percentage of TEs in removed orders:"
[1] 0
[1] "Summary of n counts"
[1] "Summary of n features"
[1] "Loci present in the annotation:"

TRUE 
3006 
[1] "young"


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "Percentage of TEs in removed orders:"
[1] 0
[1] "Summary of n counts"
[1] "Summary of n features"
[1] "Loci present in the annotation:"

 TRUE 
17410 


# STARsolo TE with EM

In [19]:
STARsolo_TE_EM_objs <- list()
samples <- c("all", "old", "young")

STARsolo_TE_path <- ""
main_path <- paste0(STARsolo_TE_path, "/", dataset_id, "/")

for (sample in samples) {
    print(sample)

    path <- paste0(main_path, sample, "/TE_Solo.out/Gene/raw/")
    filteredBarcodes <- read.table(paste0(main_path, sample, "/TE_Solo.out/Gene/filtered/barcodes.tsv"))$V1

    STARsolo_TE_EM_mat <- Seurat::ReadMtx(
        mtx = paste0(path, "UniqueAndMult-EM.mtx"),
        cells = paste0(path, "barcodes.tsv"),
        features = paste0(path, "features.tsv")
    ) # read matrix
    STARsolo_TE_EM_mat <- STARsolo_TE_EM_mat[, filteredBarcodes] # keep only barcodes passing thresholds

    STARsolo_TE_EM_objs[[sample]] <- Seurat::CreateSeuratObject(STARsolo_TE_EM_mat,
        project = paste0("STARsolo_TE_EM_", sample),
        min.cells = thrMinCells, min.features = 0
    )

    # Remove some classes of TEs (already not present in SoloTE)
    classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")
    starsoloTEs <- Features(STARsolo_TE_EM_objs[[sample]])

    filteredStarsoloTEs <- conversion_table[conversion_table$stellarscopeID %in% starsoloTEs, ] %>%
        dplyr::filter(!class %in% classesToExclude) %>%
        pull(stellarscopeID)

    print("Percentage of TEs in removed orders:")
    print(length(setdiff(starsoloTEs, filteredStarsoloTEs)) / length(starsoloTEs) * 100)

    STARsolo_TE_EM_objs[[sample]] <- STARsolo_TE_EM_objs[[sample]][intersect(starsoloTEs, filteredStarsoloTEs), ]

    print("Summary of n counts")
    print(summary(STARsolo_TE_EM_objs[[sample]]$nCount_RNA))
    print("Summary of n feature")
    print(summary(STARsolo_TE_EM_objs[[sample]]$nFeature_RNA))

    print("Loci present in the annotation:")
    print(table(rownames(STARsolo_TE_EM_objs[[sample]]) %in% conversion_table$stellarscopeID))
}

# add to the list of objects
objList[["STARsolo_TE_EM"]] <- STARsolo_TE_EM_objs

[1] "all"


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "Percentage of TEs in removed orders:"
[1] 0.005746301
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  29696   49460   55574   56187   62737   97257 
[1] "Summary of n feature"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   6268    7458    7701    7704    7976    9143 
[1] "Loci present in the annotation:"

 TRUE 
34803 
[1] "old"


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "Percentage of TEs in removed orders:"
[1] 0.03131851
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  28444   52263   59914   60686   68063  112737 
[1] "Summary of n feature"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2088    2362    2434    2427    2493    2687 
[1] "Loci present in the annotation:"

TRUE 
3192 
[1] "young"


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[1] "Percentage of TEs in removed orders:"
[1] 0
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  30052   45604   49923   50030   54333   77213 
[1] "Summary of n feature"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   7392    8256    8474    8463    8697    9818 
[1] "Loci present in the annotation:"

 TRUE 
37855 


# SoloTE Default (thr 255)

In [9]:
SoloTE_TEobjs <- list()
samples <- c("all", "old", "young")

SoloTE_path <- ""
main_path <- paste0(SoloTE_path, "/SoloTEout/", dataset_id, "/")

for (sample in samples) {
    print(sample) 
    path <- paste0(main_path, sample, "/", sample, "_SoloTE_output/", sample, "_legacytes_MATRIX/")
    # read matrix
    SoloTE_mat <- Seurat::ReadMtx(
        mtx = paste0(path, "matrix.mtx"),
        cells = paste0(path, "barcodes.tsv"),
        features = paste0(path, "features.tsv")
    )
    # filter cells
    filteredBarcodes <- colnames(splatter_objs[[sample]])
    filteredBarcodes %in% colnames(SoloTE_mat)
    SoloTE_mat <- SoloTE_mat[,filteredBarcodes]
    # select locus level TEs
    TEs <- grep("SoloTE", rownames(SoloTE_mat), value = T)
    locusTEs <- grep("chr", TEs, value = T)
    # select genes
    genes <- setdiff(rownames(SoloTE_mat), TEs)

    # subset the matrix keeping only genes
    SoloTE_GENEmat <- SoloTE_mat[genes, ]
    # create Seurat object with shallow filtering of genes expressed in at least 50 cells and cells expressing at least 100 genes
    SoloTE_GENEobj <- Seurat::CreateSeuratObject(SoloTE_GENEmat,
        project = paste0("SoloTE_unique_", sample),
        min.cells = thrMinCells, min.features = 0
    )
    print("gene obj")
    print(SoloTE_GENEobj)
    #209 genes

    # subset the matrix keeping only TEs
    SoloTE_TEmat <- SoloTE_mat[locusTEs, ]
    # remove "SoloTE" from the name of the TEs and subtitute "|" and "_" with "-"
    rownames(SoloTE_TEmat) <- gsub("SoloTE\\|", "", rownames(SoloTE_TEmat))
    rownames(SoloTE_TEmat) <- gsub("\\|", "-", rownames(SoloTE_TEmat))
    rownames(SoloTE_TEmat) <- gsub("\\_", "-", rownames(SoloTE_TEmat))
    rownames(SoloTE_TEmat) <- gsub("\\?", "", rownames(SoloTE_TEmat))

    table(rownames(SoloTE_TEmat) %in% conversion_table$soloteID)
    # transform into Stellarscope IDs
    rownames(SoloTE_TEmat) <- conversion_table$stellarscopeID[match(rownames(SoloTE_TEmat), conversion_table$soloteID)]

    # create Seurat object with shallow filtering of TEs expressed in at least thrMinCells cells and cells expressing at least 20 TEs
    SoloTE_TEobjs[[sample]] <- Seurat::CreateSeuratObject(SoloTE_TEmat,
        project = paste0("SoloTE_unique_", sample),
        min.cells = thrMinCells, min.features = 0
    )

    print("Summary of n counts")
    print(summary(SoloTE_TEobjs[[sample]]$nCount_RNA))
    print("Summary of n features")
    print(summary(SoloTE_TEobjs[[sample]]$nFeature_RNA))

    print("Loci present in the annotation:")
    print(table(rownames(SoloTE_TEobjs[[sample]]) %in% conversion_table$stellarscopeID))
}

objList[["SoloTE_unique"]] <- SoloTE_TEobjs

[1] "all"
[1] "gene obj"
An object of class Seurat 
154 features across 500 samples within 1 assay 
Active assay: RNA (154 features, 0 variable features)
 1 layer present: counts
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  20778   34062   38288   38732   42958   67021 
[1] "Summary of n features"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2281    2634    2712    2708    2788    3062 
[1] "Loci present in the annotation:"

TRUE 
6128 
[1] "old"
[1] "gene obj"
An object of class Seurat 
22 features across 500 samples within 1 assay 
Active assay: RNA (22 features, 0 variable features)
 1 layer present: counts
[1] "Summary of n counts"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  32854   55496   62526   63211   70242  112164 
[1] "Summary of n features"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2017    2279    2345    2337    2399    2591 
[1] "Loci present in the annotation:"

TRUE 
3034 
[1] "young"
[1] "gene obj"
An object of cl

# SoloTE thr 2,1 and 0 

In [21]:
SoloTE_TEobjs <- list()
samples <- c("all", "old", "young")

SoloTE_path <- ""

for(thr in c("thr2","thr1","thr0")){

    main_path <- paste0(SoloTE_path,"/SoloTEout_",thr,"/", dataset_id, "/")

    for (sample in samples){
        print(sample)
        path <- paste0(main_path, sample, "/", sample, "_SoloTE_output/", sample, "_legacytes_MATRIX/")
        # read matrix
        SoloTE_mat <- Seurat::ReadMtx(
            mtx = paste0(path, "matrix.mtx"),
            cells = paste0(path, "barcodes.tsv"),
            features = paste0(path, "features.tsv")
        )
        # filter cells
        filteredBarcodes <- colnames(splatter_objs[[sample]])

        SoloTE_mat <- SoloTE_mat[,filteredBarcodes]
        # select locus level TEs
        TEs <- grep("SoloTE", rownames(SoloTE_mat), value = T)
        locusTEs <- grep("chr", TEs, value = T)
        # select genes
        genes <- setdiff(rownames(SoloTE_mat), TEs)

        # subset the matrix keeping only genes
        SoloTE_GENEmat <- SoloTE_mat[genes, ]
        # create Seurat object with shallow filtering of genes expressed in at least 50 cells and cells expressing at least 100 genes
        SoloTE_GENEobj <- Seurat::CreateSeuratObject(SoloTE_GENEmat,
            project = paste0("SoloTE_",thr,"_", sample),
            min.cells = thrMinCells, min.features = 0
        )
        #209 genes
        
        # subset the matrix keeping only TEs
        SoloTE_TEmat <- SoloTE_mat[locusTEs, ]
        # remove "SoloTE" from the name of the TEs and subtitute "|" and "_" with "-"
        rownames(SoloTE_TEmat) <- gsub("SoloTE\\|", "", rownames(SoloTE_TEmat))
        rownames(SoloTE_TEmat) <- gsub("\\|", "-", rownames(SoloTE_TEmat))
        rownames(SoloTE_TEmat) <- gsub("\\_", "-", rownames(SoloTE_TEmat))
        rownames(SoloTE_TEmat) <- gsub("\\?", "", rownames(SoloTE_TEmat))

        table(rownames(SoloTE_TEmat) %in% conversion_table$soloteID)
        # transform into Stellarscope IDs
        rownames(SoloTE_TEmat) <- conversion_table$stellarscopeID[match(rownames(SoloTE_TEmat), conversion_table$soloteID)]

        # create Seurat object with shallow filtering of TEs expressed in at least thrMinCells cells and cells expressing at least 20 TEs
        SoloTE_TEobjs[[sample]] <- Seurat::CreateSeuratObject(SoloTE_TEmat,
            project = paste0("SoloTE_",thr,"_", sample),
            min.cells = thrMinCells, min.features = 0
        )

        summary(SoloTE_TEobjs[[sample]]$nCount_RNA)
        summary(SoloTE_TEobjs[[sample]]$nFeature_RNA)

        print("Loci present in the annotation:")
        print(table(rownames(SoloTE_TEobjs[[sample]]) %in% conversion_table$stellarscopeID))
    }

    objList[[paste0("SoloTE_",thr)]] <- SoloTE_TEobjs
}


[1] "all"
[1] "Loci present in the annotation:"

TRUE 
8675 
[1] "old"
[1] "Loci present in the annotation:"

TRUE 
3059 
[1] "young"
[1] "Loci present in the annotation:"

TRUE 
8637 
[1] "all"
[1] "Loci present in the annotation:"

 TRUE 
11686 
[1] "old"
[1] "Loci present in the annotation:"

TRUE 
3078 
[1] "young"
[1] "Loci present in the annotation:"

 TRUE 
13116 
[1] "all"
[1] "Loci present in the annotation:"

 TRUE 
33662 
[1] "old"
[1] "Loci present in the annotation:"

TRUE 
3271 
[1] "young"
[1] "Loci present in the annotation:"

 TRUE 
42536 


# Stellarscope

In [30]:
Stellarscope_objs <- list()
samples <- c("all", "old", "young")

Stellarscope_path <- ""
main_path <- paste0(Stellarscope_path, "/stellarscope_out/", dataset_id, "/")

for (sample in samples){
    print(sample)
    path <- paste0(main_path, sample, "/pseudobulk/")
    # read matrix
    Stellarscope_mat <- Seurat::ReadMtx(mtx = paste0(path, sample, "_pseudobulk-TE_counts.mtx"), 
                                cells = paste0(path, sample,"_pseudobulk-barcodes.tsv"), 
                                features = paste0(path, sample,"_pseudobulk-features.tsv"), 
                                feature.column = 1) # read matrix
    # remove the unassigned counts
    no_feature <- Stellarscope_mat["__no_feature",]
    Stellarscope_mat <- Stellarscope_mat[setdiff(rownames(Stellarscope_mat),"__no_feature"),]
    # substitute "_" with "-"
    rownames(Stellarscope_mat) <- gsub(pattern = "_", replacement = "-", rownames(Stellarscope_mat))

    # filter cells passing filters
    filteredBarcodes <- colnames(splatter_objs[[sample]])
    Stellarscope_mat <- Stellarscope_mat[,filteredBarcodes]

    # create Seurat object with shallow filtering of TEs expressed in at least 5% of cells and cells expressing at least 20 TEs 
    Stellarscope_objs[[sample]] <- Seurat::CreateSeuratObject(Stellarscope_mat, project = paste0("Stellarscope_", sample),
                            min.cells = thrMinCells, min.features = 0) 

    # Remove some classes of TEs (already not present in SoloTE)
    classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")
    stellarscopeTEs <- Features(Stellarscope_objs[[sample]])

    filteredStellarscopeTEs <- conversion_table[conversion_table$stellarscopeID %in% stellarscopeTEs,] %>% 
        filter(! class %in% classesToExclude) %>% pull(stellarscopeID)
    print("percentage of loci removed")
    print(length(setdiff(stellarscopeTEs, filteredStellarscopeTEs))/ length(stellarscopeTEs) * 100) # percentage removed

    Stellarscope_objs[[sample]]  <- Stellarscope_objs[[sample]][intersect(stellarscopeTEs, filteredStellarscopeTEs),]
    
    print("n count summary")
    print(summary(Stellarscope_objs[[sample]]$nCount_RNA))
    print("n features summary")
    print(summary(Stellarscope_objs[[sample]]$nFeature_RNA))

    print("Loci present in the annotation:")
    print(table(rownames(Stellarscope_objs[[sample]]) %in% conversion_table$stellarscopeID))
}

objList[["Stellarscope"]] <- Stellarscope_objs

[1] "all"
[1] "percentage of loci removed"
[1] 0
[1] "n count summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  21386   34925   39222   39690   44043   68459 
[1] "n features summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2421    2813    2887    2884    2969    3270 
[1] "Loci present in the annotation:"

TRUE 
6959 
[1] "old"
[1] "percentage of loci removed"
[1] 0
[1] "n count summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  32672   55257   62393   62992   70016  111822 
[1] "n features summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   1983    2255    2319    2312    2376    2567 
[1] "Loci present in the annotation:"

TRUE 
2991 
[1] "young"
[1] "percentage of loci removed"
[1] 0
[1] "n count summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   7816   11040   11866   11962   12866   18540 
[1] "n features summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   1329    1431    1454    1452    1473    1558 
[1] "Loci present in

# Stellarscope with posterior threshold

In [ ]:
Stellarscope_objs_thr <- list()
samples <- c("all", "old", "young")
thr <- 0.9

Stellarscope_path <- ""
main_path <- paste0(Stellarscope_path,"/stellarscope_out/", dataset_id, "/")

for (sample in samples){
    print(sample)
    path <- paste0(main_path, sample, "/pseudobulk_thr", thr,"/")
    # read matrix
    Stellarscope_mat <- Seurat::ReadMtx(mtx = paste0(path, sample, "_pseudobulk-TE_counts.mtx"), 
                                cells = paste0(path, sample,"_pseudobulk-barcodes.tsv"), 
                                features = paste0(path, sample,"_pseudobulk-features.tsv"), 
                                feature.column = 1) # read matrix
    # remove the unassigned counts
    no_feature <- Stellarscope_mat["__no_feature",]
    Stellarscope_mat <- Stellarscope_mat[setdiff(rownames(Stellarscope_mat),"__no_feature"),]
    # substitute "_" with "-"
    rownames(Stellarscope_mat) <- gsub(pattern = "_", replacement = "-", rownames(Stellarscope_mat))

    # filter cells passing filters
    filteredBarcodes <- colnames(splatter_objs[[sample]])
    Stellarscope_mat <- Stellarscope_mat[,filteredBarcodes]

    # create Seurat object with shallow filtering of TEs expressed in at least 5% of cells and cells expressing at least 20 TEs 
    Stellarscope_objs_thr[[sample]] <- Seurat::CreateSeuratObject(Stellarscope_mat, project = paste0("Stellarscope_thr09_", sample),
                            min.cells = thrMinCells, min.features = 0) 

    # Remove some classes of TEs (already not present in SoloTE)
    classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")
    stellarscopeTEs <- Features(Stellarscope_objs_thr[[sample]])

    filteredStellarscopeTEs <- conversion_table[conversion_table$stellarscopeID %in% stellarscopeTEs,] %>% 
        filter(! class %in% classesToExclude) %>% pull(stellarscopeID)
    print("percentage of loci removed")
    print(length(setdiff(stellarscopeTEs, filteredStellarscopeTEs))/ length(stellarscopeTEs) * 100) # percentage removed

    Stellarscope_objs_thr[[sample]]  <- Stellarscope_objs_thr[[sample]][intersect(stellarscopeTEs, filteredStellarscopeTEs),]
    
    print("n count summary")
    print(summary(Stellarscope_objs_thr[[sample]]$nCount_RNA))
    print("n features summary")
    print(summary(Stellarscope_objs_thr[[sample]]$nFeature_RNA))

    print("Loci present in the annotation:")
    print(table(rownames(Stellarscope_objs_thr[[sample]]) %in% conversion_table$stellarscopeID))
}

objList[[paste0("Stellarscope_thr09")]] <- Stellarscope_objs_thr

[1] "all"
[1] "percentage of loci removed"
[1] 0
[1] "n count summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  21157   34676   38990   39447   43818   68177 
[1] "n features summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   2268    2649    2728    2723    2807    3090 
[1] "Loci present in the annotation:"

TRUE 
6094 
[1] "old"
[1] "percentage of loci removed"
[1] 0
[1] "n count summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  32644   55197   62316   62910   69911  111651 
[1] "n features summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   1983    2255    2319    2312    2375    2565 
[1] "Loci present in the annotation:"

TRUE 
2989 
[1] "young"
[1] "percentage of loci removed"
[1] 0
[1] "n count summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   7210   10347   11195   11258   12168   17639 
[1] "n features summary"
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   1027    1117    1136    1135    1153    1224 
[1] "Loci present in

## Normalize objects

In [ ]:

for (sample in samples){
    for (tool in names(objList)){
        objList[[tool]][[sample]] <- NormalizeData(objList[[tool]][[sample]], 
            normalization.method = "LogNormalize", scale.factor = 10000)
    }
}

Normalizing layer: counts



Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts

Normalizing layer: counts



# Save objects

In [33]:
dir.create("workspaces")
save.image("workspaces/mm_RA_simulation_evaluation_objectCreation.Rdata")


Warning message in dir.create("workspaces"):
“'workspaces' already exists”
